# 121. PDF 원인 및 모든8개 반복누락 감사
가이드11쪽 그림3를 원문과 이미지로 확인. 같은구간 정상 계량잔차도 함께 집계해 날짜별방향을 불량근거로 잘못확정하지 않음.113~120 내부선택 예측에대해574개/미성형8개 중복검증. 사후설명전용, 피처·임계값에 새로반영한 검증이아님.


In [1]:
from pathlib import Path
import pandas as pd,numpy as np,pymupdf
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
p=pymupdf.open(root/'data/04. Guidebook_Molding.pdf')
print('GUIDE PAGE11',p[10].get_text())
image_dir=root/'reports/figures';image_dir.mkdir(exist_ok=True)
p[10].get_pixmap(matrix=pymupdf.Matrix(1.5,1.5)).save(image_dir/'rg3_shortshot_guide_page11.png')
r=pd.read_csv(t/'rg3_shortshot_unlabeled_teacher_residuals.csv',parse_dates=['TimeStamp'])
r.groupby(['block','actual']).agg(rows=('recovery','size'),recovery_median=('recovery','median'),recovery_positive=('recovery',lambda s:(s>0).sum())).reset_index().to_csv(t/'rg3_shortshot_teacher_block_confounding.csv',index=False,encoding='utf-8-sig')
models={
 '113':('rg3_shortshot_specialist_predictions.csv',None),
 '114':('rg3_shortshot_case_contrast_predictions.csv','inner_F1'),
 '116':('rg3_shortshot_unlabeled_teacher_predictions.csv','inner_F1'),
 '117':('rg3_shortshot_recovery_modes_predictions.csv','inner_F1'),
 '118':('rg3_shortshot_causal_distribution_predictions.csv','inner_F1'),
 '119':('rg3_shortshot_causal_teacher_predictions.csv','inner_F1'),
 '120':('rg3_shortshot_causal_fullhistory_predictions.csv','inner_F1')}
case=None
for name,(file,rule) in models.items():
 s=pd.read_csv(t/file,parse_dates=['TimeStamp'])
 s=s[s.model.eq('primary_inner_selected')]
 if rule:s=s[s.rule.eq(rule)]
 assert len(s)==574 and s.TimeStamp.is_unique
 s=s[s.actual.eq(1)][['TimeStamp','prediction','score','threshold']].rename(columns={v:name+'_'+v for v in ['prediction','score','threshold']})
 assert len(s)==8
 case=s if case is None else case.merge(s,on='TimeStamp',validate='one_to_one')
case.to_csv(t/'rg3_shortshot_iterations_all8.csv',index=False,encoding='utf-8-sig')
print(case[['TimeStamp']+[v+'_prediction' for v in models]].to_string(index=False))
print('LAST METRICS',pd.read_csv(t/'rg3_shortshot_causal_fullhistory_metrics.csv').to_string(index=False))


GUIDE PAGE11 「사출성형기 AI 데이터셋」 분석실습 가이드북
10
게 하면 취출 후 경시 변화에 의한 평면 밀도를 이루면서 왜곡(Distortion), 변형
(Warpage), 또는 균열(Cracking) 이 발생하게 된다. 
2) 이슈사항(Pain point)
● 공정의 문제 상황 
- 사출성형에서는 각종 성형불량이 발생한다. 이들 불량현상을 해결하기 위해서는 우선 
현상자체를 물리·화학적으로 잘 이해해 둘 필요가 있지만 개개의 경우에 대해서 신속
히 그 원인을 규명해서 적절한 대책을 세우는 것은 쉬운 일이 아니다. 원인이 되는 요
인이 몇 가지가 복잡하게 동시에 얽혀서 나오기도 하고, 한 가지 불량현상의 대책을 세
우는 것에 의해서 다른 불량현상을 해결하는 경우도 있을 것이다. 불량을 발생시키는 
중요한 인자에 대한 이해가 반드시 필요한 부분이라 하겠다. 하여 이하에 사출성형의 
중요한 불량현상과 그 직접적인 원인에 대해서 설명해 그들 원인에 직결하는 요인을 
대략적으로 정리하였으며 실제 공정에서의 이슈사항들을 같이 나열하였다.
● 불량현상과 원인
a. 쇼트숏
◦ 쇼트숏(Shot Shot)은 금형 내에서의 원료 플라스틱의 충전량이 부족함으로 인해 
발생하는 상태이다. 게이트로부터 떨어진 부분에서 발생하는 잔물결의 주름도 쇼트
숏 현상이다. 직접적인 원인은 원료 공급불량, 충전압력 부족, 금형 내 유동중의 원
료고화, 금형 내의 공기 저항 등으로 분류할 수 있다. 이들 원인은 싱크마크, 플로우
마크 등의 현상원인과 공통으로, 특히 싱크마크와 쇼트숏은 현상적으로 구별하기 
어려운 일도 있다.
[그림 3] 불량현상 - 쇼트숏



          TimeStamp  113_prediction  114_prediction  116_prediction  117_prediction  118_prediction  119_prediction  120_prediction
2020-10-22 01:09:23               0               0               1               0               0               0               0
2020-10-22 04:48:39               0               0               1               0               1               0               0
2020-10-22 05:42:13               0               0               1               0               1               1               1
2020-10-22 06:12:03               0               0               1               0               0               1               0
2020-10-22 06:58:22               0               0               1               0               0               1               1
2020-10-23 02:25:54               0               0               1               0               0               0               0
2020-10-23 05:47:40               0               0               1         